# Jawad Hassan
# 2230-0035
# BS AI
# ANN
# Lab 08


### Task:
### RNN WITH IMBD DATASET

In [1]:
from google.colab import files
uploaded = files.upload()

!pip install kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip imdb-dataset-of-50k-movie-reviews.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:00<00:00, 188MB/s]

Archive:  imdb-dataset-of-50k-movie-reviews.zip
  inflating: IMDB Dataset.csv        


In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
df = pd.read_csv('IMDB Dataset.csv')
X = df['review'].values
y = np.where(df['sentiment'] == 'positive', 1, 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad = pad_sequences(X_train_seq, maxlen=200)
X_test_pad = pad_sequences(X_test_seq, maxlen=200)
model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=128, input_length=200))
model.add(SimpleRNN(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2))
model.add(SimpleRNN(32, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model.fit(X_train_pad, y_train, epochs=20, batch_size=64, validation_split=0.2, callbacks=[early_stop])
loss, accuracy = model.evaluate(X_test_pad, y_test)
print(accuracy)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


500/500 ━━━━━━━━━━━━━━━━━━━━ 25s 34ms/step - accuracy: 0.5030 - loss: 0.7081 - val_accuracy: 0.5149 - val_loss: 0.6930
Epoch 2/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 32ms/step - accuracy: 0.5023 - loss: 0.6958 - val_accuracy: 0.5129 - val_loss: 0.6928
Epoch 3/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 32ms/step - accuracy: 0.5042 - loss: 0.6936 - val_accuracy: 0.5026 - val_loss: 0.6930
Epoch 4/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - accuracy: 0.5114 - loss: 0.6933 - val_accuracy: 0.5347 - val_loss: 0.6918
Epoch 5/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 32ms/step - accuracy: 0.5257 - loss: 0.6900 - val_accuracy: 0.5775 - val_loss: 0.6712
Epoch 6/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 32ms/step - accuracy: 0.5754 - loss: 0.6678 - val_accuracy: 0.6069 - val_loss: 0.6436
Epoch 7/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 32ms/step - accuracy: 0.6996 - loss: 0.5805 - val_accuracy: 0.7770 - val_loss: 0.4866
Epoch 8/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 32ms/step - accuracy: 0.7653 - loss: 0.5047 - val_accurac

In [19]:
def predict_sentiment(review_text):
    seq = tokenizer.texts_to_sequences([review_text])
    pad = pad_sequences(seq, maxlen=200)
    prediction = model.predict(pad)[0][0]
    if prediction >= 0.5:
        return "Positive", prediction
    else:
        return "Negative", prediction
custom_review = "The cinematography was ok but the pacing felt incredibly slow and disjointed. "
sentiment, score = predict_sentiment(custom_review)
print(sentiment, score)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Negative 0.25132436
